In [1]:
# 1. Install Unsloth (The core engine)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# 2. Install the Hugging Face Ecosystem
!pip install transformers datasets trl peft accelerate bitsandbytes

# 3. Install xformers (Memory optimization)
!pip install xformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-smsccxcb/unsloth_0e917d631a614c4b8474a6fb1f5dabe2
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-smsccxcb/unsloth_0e917d631a614c4b8474a6fb1f5dabe2
  Resolved https://github.com/unslothai/unsloth.git to commit 07abcb46de55d6fdbc2f5308be5a2a9a74d32d26
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 99.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.9/402.9 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 106.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 24.0 MB/s eta 0:00:00

In [2]:
import torch
from unsloth import FastLanguageModel

# 1. Load the Base Model in 4-bit Quantization
print("Loading Model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048, # How much context the model can remember at once
    dtype = None, # Auto-detects the best format for your GPU
    load_in_4bit = True, # Shrinks the model to fit in the free GPU
)

# 2. Add the LoRA Adapters (This is the part we will actually train)
print("Applying LoRA Adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank: Size of the adapter
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Connecting to all attention layers
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves massive amounts of VRAM
    random_state = 3407,
)

print(f"\n✅ SUCCESS! Connected to GPU: {torch.cuda.get_device_name(0)}")
print(f"✅ Model Llama-3.2-3B is locked, loaded, and ready to fine-tune!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading Model...
==((====))==  Unsloth 2026.3.15: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

chat_template.jinja: 0.00B [00:00, ?B/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.


Applying LoRA Adapters...


Unsloth 2026.3.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.



✅ SUCCESS! Connected to GPU: Tesla T4
✅ Model Llama-3.2-3B is locked, loaded, and ready to fine-tune!


In [3]:
from datasets import load_dataset

# 1. Define the exact structure the model will learn to read
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# 2. Add the "Stop Talking" token (Crucial!)
EOS_TOKEN = tokenizer.eos_token # Without this, the model will generate endless gibberish

# 3. Create the formatting function
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []

    for instruction, input, output in zip(instructions, inputs, outputs):
        # We inject the data into our template and cap it with the EOS token
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 4. Download and process the data
print("Downloading dataset...")
dataset = load_dataset("yahma/alpaca-cleaned", split = "train")

# Here we will only use the first 2,000 rows to save time
dataset = dataset.select(range(2000))

print("Formatting data...")
dataset = dataset.map(formatting_prompts_func, batched = True)

# 5. Sanity Check: Let's look at the very first row
print("\n✅ Dataset Ready! Here is exactly what the model will see for row 1:\n")
print(dataset[0]["text"])

README.md: 0.00B [00:00, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Formatting data...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


✅ Dataset Ready! Here is exactly what the model will see for row 1:

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy g

In [4]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configuring the Trainer...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # We are doing a fast sprint of 60 steps!
        learning_rate = 2e-4, # Standard learning rate for LoRA
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit", # Uses less memory
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Disabling WandB logging for this quick sprint
    ),
)

print("Starting the Engine. Training commencing now...\n")
trainer_stats = trainer.train()

print(f"\n✅ TRAINING COMPLETE!")

Configuring the Trainer...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/2000 [00:00<?, ? examples/s]

Starting the Engine. Training commencing now...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Step,Training Loss
1,2.275709
2,1.694321
3,1.645426
4,1.496662
5,1.667165
6,1.523277
7,1.510720
8,1.203549
9,1.353878
10,1.155146



✅ TRAINING COMPLETE!


In [5]:
from transformers import TextStreamer

# 1. Format your prompt
prompt = alpaca_prompt.format(
    "Explain the concept of smartphone in one simple paragraph.", # Instruction
    "", # Input
    "", # Response (Left blank)
)

# 2. Tokenize and send to GPU
inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

# 3. Set up the Streamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True, skip_special_tokens = True)

# 4. Generate the output!
print("🧠 Model is thinking...\n")
_ = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 128,
    pad_token_id = tokenizer.eos_token_id,
    use_cache = False # <--- THE FIX: Bypassing the KV-Cache bug
)

Both `max_new_tokens` (=128) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


🧠 Model is thinking...



/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

A smartphone 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


is a portable electronic device that combines the functions of a computer, a camera, a phone, and a personal assistant, all in one small device. It is used for communication, entertainment, and information gathering. Smartphones are connected to the internet and have various apps, games, and multimedia content. They can be used to make phone calls, send text messages, browse the internet, take photos and videos, and access various services and apps.


In [6]:
from huggingface_hub import login
login() # Paste your 'Write' token here when prompted


In [7]:
# Replace 'your-username' with your actual Hugging Face username
model.push_to_hub("kumarsarthak98/llama-3.2-3b-agentic-sprint", token = True)
tokenizer.push_to_hub("kumarsarthak98/llama-3.2-3b-agentic-sprint", token = True)

print("🚀 Model is live on the Hub!")

README.md:   0%|          | 0.00/572 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

Saved model to https://huggingface.co/kumarsarthak98/llama-3.2-3b-agentic-sprint


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mplymcwti_/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


🚀 Model is live on the Hub!


In [8]:
!pip install gradio
import gradio as gr

In [9]:
def generate_response(instruction, user_input=""):
    # 1. Format using the Alpaca template we used for training
    prompt = alpaca_prompt.format(instruction, user_input, "")

    # 2. Tokenize
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # 3. Generate (using the use_cache=False fix we found earlier)
    outputs = model.generate(
        **inputs,
        max_new_tokens = 128,
        pad_token_id = tokenizer.eos_token_id,
        use_cache = False
    )

    # 4. Decode and clean up the output
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]

    # Remove the prompt part and only return the "Response"
    response = decoded.split("### Response:")[-1].strip()
    return response

In [ ]:
import gradio as gr

def generate_response(instruction, user_input=""):
    prompt = alpaca_prompt.format(instruction, user_input, "")
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    # We use a shorter max_new_tokens for the demo to keep it snappy
    outputs = model.generate(
        **inputs,
        max_new_tokens = 256, # Increased this so you get longer answers
        pad_token_id = tokenizer.eos_token_id,
        use_cache = False
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    response = decoded.split("### Response:")[-1].strip()
    return response

# --- IMPROVED UI LAYOUT ---
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🤖 My Agentic AI Fine-Tune")
    gr.Markdown("Fine-tuned Llama-3.2-3B. Optimized for instruction following.")

    with gr.Row():
        with gr.Column():
            instr = gr.Textbox(
                label="Instruction",
                placeholder="What should the AI do?",
                lines=3 # Makes the input box taller
            )
            ctx = gr.Textbox(
                label="Input/Context (Optional)",
                placeholder="Paste background data here...",
                lines=5
            )
            submit_btn = gr.Button("Generate Response", variant="primary")

        with gr.Column():
            # THIS IS THE BOX YOU WANTED TO INCREASE
            out = gr.Textbox(
                label="AI Response",
                lines=15,       # Sets the initial height to 15 lines
                max_lines=30,   # Allows it to grow up to 30 lines before scrolling
                interactive=False,
                show_copy_button=True # Great for user experience
            )

    # Link the button to the function
    submit_btn.click(fn=generate_response, inputs=[instr, ctx], outputs=out)

demo.launch(share=True)

/tmp/ipykernel_2178/3062230053.py:20: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme="soft") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d2282a7ce8748d7212.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [12]:
%%writefile app.py
import gradio as gr
import torch
from unsloth import FastLanguageModel

# 1. Load the model and tokenizer (use the same settings as your sprint)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Define the prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

# 3. Define the generation function
def generate_response(instruction, user_input=""):
    prompt = alpaca_prompt.format(instruction, user_input, "")
    inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens = 256,
        pad_token_id = tokenizer.eos_token_id,
        use_cache = False
    )

    decoded = tokenizer.batch_decode(outputs, skip_special_tokens = True)[0]
    response = decoded.split("### Response:")[-1].strip()
    return response

# 4. Build the Gradio Interface
with gr.Blocks(theme="soft") as demo:
    gr.Markdown("# 🤖 My Agentic AI Fine-Tune")
    with gr.Row():
        with gr.Column():
            instr = gr.Textbox(label="Instruction", lines=3)
            ctx = gr.Textbox(label="Input/Context (Optional)", lines=5)
            submit_btn = gr.Button("Generate", variant="primary")
        with gr.Column():
            out = gr.Textbox(label="AI Response", lines=15)

    submit_btn.click(fn=generate_response, inputs=[instr, ctx], outputs=out)

# 5. Launch (Note: No 'share=True' needed for HF Spaces)
demo.launch()

Overwriting app.py
